<a href="https://colab.research.google.com/github/jonathan9970/jonathan9970.github.io/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
ESP32-CAM MJPEG Stream Viewer + DEFORESTATION Detection (YOLOv8)
---------------------------------------------------------------

Requirements:
    pip install requests opencv-python numpy ultralytics

Controls:
    Q = Quit
    D = Turn detection ON/OFF
    + = Increase sensitivity
    - = Decrease sensitivity
"""

import cv2
import numpy as np
import requests
import time
from ultralytics import YOLO


# ---------------- CONFIG ----------------

ESP32_IP = "10.0.0.2"   # CHANGE THIS TO YOUR ESP32-CAM IP
STREAM_URL = f"http://{ESP32_IP}/stream"
TIMEOUT = 10

MODEL_NAME = "yolov8n.pt"   # use custom deforestation model later if trained
CONF_THRESH = 0.30          # lower = more detections

DETECT_EVERY = 1


# These are example classes.
# For real deforestation detection, you would train a custom YOLO model
# with classes like: cut_tree, fallen_log, smoke, fire, cleared_land.
FOREST_CLASSES = None
# Set to None so YOLO can detect all possible objects.


# ---------------- READ ESP32 STREAM ----------------

def read_mjpeg_stream(url: str, timeout: int = 10):
    """Yields frames from ESP32 MJPEG stream."""
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()

        buf = b""

        for chunk in r.iter_content(chunk_size=4096):
            if not chunk:
                continue

            buf += chunk

            while True:
                start = buf.find(b"\xff\xd8")
                end = buf.find(b"\xff\xd9", start)

                if start == -1 or end == -1:
                    break

                jpg = buf[start:end + 2]
                buf = buf[end + 2:]

                arr = np.frombuffer(jpg, dtype=np.uint8)
                frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)

                if frame is not None:
                    yield frame


# ---------------- SIMPLE GREEN COVERAGE CHECK ----------------

def green_coverage_check(frame):
    """Checks how much green vegetation is in the image."""
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower_green = np.array([35, 40, 40])
    upper_green = np.array([85, 255, 255])

    mask = cv2.inRange(hsv, lower_green, upper_green)

    green_pixels = cv2.countNonZero(mask)
    total_pixels = frame.shape[0] * frame.shape[1]

    green_percent = (green_pixels / total_pixels) * 100

    return green_percent


# ---------------- DRAW YOLO DETECTIONS ----------------

def draw_detections(frame, results, names):
    """Draw bounding boxes for detected objects."""
    count = 0

    for box in results[0].boxes:
        conf = float(box.conf[0])

        if conf < CONF_THRESH:
            continue

        class_id = int(box.cls[0])
        label_name = names[class_id]

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        color = (0, 0, 255)

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        label = f"Threat ({label_name}) {conf:.0%}"

        (tw, th), _ = cv2.getTextSize(
            label,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            2
        )

        cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw, y1), color, -1)

        cv2.putText(
            frame,
            label,
            (x1, y1 - 4),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )

        count += 1

    return count


# ---------------- DRAW SCREEN DISPLAY ----------------

def draw_hud(frame, fps, threat_count, green_percent, detection_on):
    """Top overlay information."""
    h, w = frame.shape[:2]

    overlay = frame.copy()

    cv2.rectangle(overlay, (0, 0), (w, 70), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    status = "ON" if detection_on else "OFF"

    if green_percent < 30:
        forest_status = "POSSIBLE DEFORESTATION"
        status_color = (0, 0, 255)
    else:
        forest_status = "FOREST LOOKS NORMAL"
        status_color = (0, 255, 100)

    cv2.putText(
        frame,
        f"FPS: {fps:.1f}",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 100),
        2
    )

    cv2.putText(
        frame,
        f"Threats: {threat_count}",
        (130, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 200, 255),
        2
    )

    cv2.putText(
        frame,
        f"Detect: {status}",
        (300, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 200, 255),
        2
    )

    cv2.putText(
        frame,
        f"Green Coverage: {green_percent:.1f}%",
        (10, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        forest_status,
        (300, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        status_color,
        2
    )


# ---------------- MAIN PROGRAM ----------------

def main():
    global CONF_THRESH

    print("Loading YOLO model...")
    model = YOLO(MODEL_NAME)
    names = model.names

    print("Connecting to ESP32-CAM stream...")
    print("Open stream:", STREAM_URL)

    detection_on = True
    frame_count = 0
    fps = 0
    start_time = time.time()
    last_results = None

    for frame in read_mjpeg_stream(STREAM_URL, TIMEOUT):

        frame_count += 1

        # FPS calculation
        if time.time() - start_time >= 1:
            fps = frame_count / (time.time() - start_time)
            frame_count = 0
            start_time = time.time()

        # Green coverage detection
        green_percent = green_coverage_check(frame)

        # AI detection
        if detection_on:
            last_results = model(
                frame,
                conf=CONF_THRESH,
                classes=FOREST_CLASSES,
                verbose=False
            )

        # Draw detections
        threat_count = 0

        if detection_on and last_results is not None:
            threat_count = draw_detections(frame, last_results, names)

        # Draw screen information
        draw_hud(frame, fps, threat_count, green_percent, detection_on)

        cv2.imshow("ESP32 Deforestation Detection Robot", frame)

        # Controls
        key = cv2.waitKey(1) & 0xFF

        if key in (27, ord("q")):
            break

        elif key == ord("d"):
            detection_on = not detection_on
            print("Detection:", "ON" if detection_on else "OFF")

        elif key == ord("+"):
            CONF_THRESH = min(0.9, CONF_THRESH + 0.05)
            print("Confidence threshold:", CONF_THRESH)

        elif key == ord("-"):
            CONF_THRESH = max(0.1, CONF_THRESH - 0.05)
            print("Confidence threshold:", CONF_THRESH)

    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()